<a href="https://colab.research.google.com/github/Grecia1225/data-science-projects/blob/main/project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install transformers datasets scikit-learn torch -q

In [10]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import classification_report
from tqdm import tqdm
print("Version", torch.__version__)
print("GPU there:", torch.cuda.is_available())

Version 2.11.0+cu128
GPU there: True


In [4]:
from datasets import load_dataset
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
print("Dataset loaded!")
print(f"Training tweets: {len(dataset['train'])}")
print(f"Validation tweets: {len(dataset['validation'])}")
print(f"Test tweets: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset loaded!
Training tweets: 45615
Validation tweets: 2000
Test tweets: 12284


In [5]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
class TweetDataset(Dataset):
    def __init__(self, data):
        self.texts = data["text"]
        self.labels = data["label"]
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx])
        }
train_data = TweetDataset(dataset["train"])
val_data   = TweetDataset(dataset["validation"])
test_data  = TweetDataset(dataset["test"])
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=16)
test_loader  = DataLoader(test_data,  batch_size=16)
print("Data ready!")
print(f"Training batches: {len(train_loader)}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Data ready!
Training batches: 2851


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
model = model.to(device)
print(f"BERT loaded and running on: {device}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT loaded and running on: cuda


In [7]:
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} done — loss: {avg_loss:.4f}")
print("\nTraining complete!")

Epoch 1/3: 100%|██████████| 2851/2851 [18:10<00:00,  2.61it/s]


Epoch 1 done — loss: 0.6523


Epoch 2/3: 100%|██████████| 2851/2851 [18:13<00:00,  2.61it/s]


Epoch 2 done — loss: 0.4707


Epoch 3/3: 100%|██████████| 2851/2851 [18:13<00:00,  2.61it/s]

Epoch 3 done — loss: 0.2938

Training complete!


In [12]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(
    all_labels,
    all_preds,
    target_names=["negative", "neutral", "positive"]
))

Evaluating: 100%|██████████| 125/125 [00:17<00:00,  6.99it/s]

              precision    recall  f1-score   support

    negative       0.64      0.68      0.66       312
     neutral       0.73      0.68      0.70       869
    positive       0.76      0.80      0.78       819

    accuracy                           0.73      2000
   macro avg       0.71      0.72      0.71      2000
weighted avg       0.73      0.73      0.72      2000



In [13]:
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=128,
        padding="max_length",
        truncation=True
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    labels = ["negative", "neutral", "positive"]
    print(f"Text: {text}")
    print(f"Sentiment: {labels[pred]}\n")

predict("This is the best thing I have ever bought!")
predict("I am so disappointed with this product.")
predict("The package arrived today.")

Text: This is the best thing I have ever bought!
Sentiment: positive

Text: I am so disappointed with this product.
Sentiment: negative

Text: The package arrived today.
Sentiment: neutral



In [22]:
predict("i drank water")

Text: i drank water
Sentiment: neutral



In [23]:
predict("i drank 5 liters thumbsup")

Text: i drank 5 liters thumbsup
Sentiment: positive

